# Train a predictor

One notebook for both modes. Set `mode` and the feature spec in the CONFIG cell; everything downstream is shared.

* **parallel** -> teacher forcing on flattened `(sample, feature)`.
* **autoregressive** -> rollout on `(case, time, feature)`; the first `n_ar` columns are the state fed back.
* `predict_tendency=True` makes the net predict an increment added to the previous state *inside* `_step` (residual skip; needs a state column, i.e. `n_ar >= 1`), and it is scored in state space.
* `target_tendency=True` instead makes the **target** the increment `dM = M_{i+1}-M_i`, scored in tendency space (normalized by `std(dM)`, so persistence is not a free win) and reconstructed to a state trajectory by cumulative sum at evaluation — so drift shows. Mutually exclusive with `predict_tendency`.


In [ ]:
from data import DatasetMaker, FeatureSpec, FeatureSelector, compute_norm_stats
from model import FCNN, LinearRegression, ParallelPredictor, AutoregressivePredictor
from metrics import normalize, real_units, invert_transform, compute_rmse
from experiment import PredictorConfig
from plots import scatter, loss_trajectory

import torch
import torch.utils.data as Data
from torch import optim, nn
import numpy as np
import os
import matplotlib.pyplot as plt

random_state = 123
np.random.seed(random_state)
torch.manual_seed(random_state)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
data_dir = '../m_star_dataset'
train_ds = DatasetMaker.load(f'{data_dir}/ePBL_paper_expanded_2283_corrected_transient.nc')
val_ds   = DatasetMaker.load(f'{data_dir}/ePBL_paper_validation_2283_corrected_transient.nc')

## CONFIG

In [ ]:
### experiment identity
root = 'predictors'
run_id = 'predictor_1'
run_dir = os.path.join(root, run_id)

mode = 'parallel'            # 'parallel' | 'autoregressive'
predict_tendency = False     # residual skip in _step (needs a state column, n_ar >= 1)
target_tendency = True       # train on dM = M_{i+1}-M_i, rebuild M by cumulative sum
k = 4                        # autoregressive rollout window (ignored if parallel)

### feature spec  (ar channels come first; the first n_ar are fed back in AR mode)
# parallel dM predictor: target is the next state M_{i+1} (lag=-1); target_tendency
# trains on the increment M_{i+1}-M_i and reconstruction (cell 14 / restore)
# integrates it, so drift is visible. dM is normalized by std(dM), not std(M), so
# the net can't hide behind persistence.
ar_features = []
forcings = [FeatureSpec(n) for n in
            ['M', 'f', 'u_star', 'B', 'bl_tke', 'surf_mag', 'surf_angle', 'SS_bl', 'NN_bl']]
target = FeatureSpec('M', lag=-1)     # next-step state; differenced to dM by target_tendency

# --- parallel instantaneous state predictor -----------------------------
# predict_tendency, target_tendency = False, False
# ar_features = []
# forcings = [FeatureSpec(n) for n in ['f','u_star','B','bl_tke','surf_mag','surf_angle','SS_bl','NN_bl']]
# target = FeatureSpec('M', lag=0)                      # predict M directly

# --- autoregressive (drift-controlled) ----------------------------------
# mode, predict_tendency, target_tendency = 'autoregressive', True, False
# ar_features = [FeatureSpec('M', lag=0)]               # fed back during rollout
# forcings = [FeatureSpec(n) for n in ['f','u_star','B','bl_tke','surf_mag','surf_angle','SS_bl','NN_bl']]
# target = FeatureSpec('M', lag=-1)                     # next-step state

### architecture
network = 'fcnn'
n_neurons_list = [32, 32, 32]
activation = 'tanh'

### training
learning_rate = 2e-4
weight_decay = 5e-4
loss = 'mse_loss'
n_epochs = 3000
train_batch_size = 87277
test_batch_size = 46548

## Select features -> tensors

In [ ]:
selector_mode = 'sequence' if mode == 'autoregressive' else 'parallel'
sel = FeatureSelector(ar_features, forcings, target, mode=selector_mode,
                      target_tendency=target_tendency)
train_sel = sel.select(train_ds)
val_sel = sel.select(val_ds)
n_ar = train_sel.n_ar
feature_names = train_sel.feature_names
print('features:', feature_names, '-> target', train_sel.target_names)
print('X', tuple(train_sel.X.shape), 'y', tuple(train_sel.y.shape), 'n_ar', n_ar)

In [ ]:
# normalization stats from TRAIN only; reuse for val
fmean, fstd = compute_norm_stats(train_sel.X.numpy())
fmean, fstd = torch.tensor(fmean).float(), torch.tensor(fstd).float()
tmean = train_sel.y.reshape(-1, train_sel.y.shape[-1]).mean(0)
tstd = train_sel.y.reshape(-1, train_sel.y.shape[-1]).std(0)

X_train = normalize(train_sel.X, fmean, fstd)
X_val = normalize(val_sel.X, fmean, fstd)
y_train = normalize(train_sel.y, tmean, tstd)
y_val = normalize(val_sel.y, tmean, tstd)

train_loader = Data.DataLoader(Data.TensorDataset(X_train, y_train),
                               batch_size=train_batch_size, shuffle=True, pin_memory=True)
test_loader = Data.DataLoader(Data.TensorDataset(X_val, y_val),
                              batch_size=test_batch_size, shuffle=False, pin_memory=True)

## Build network + predictor

In [ ]:
act = {'relu': nn.ReLU(), 'tanh': nn.Tanh()}[activation]
out_size = train_sel.y.shape[-1]        # one output per target column (dM, or state)
if network == 'fcnn':
    net = FCNN(len(feature_names), n_neurons_list, act, output_size=out_size)
else:
    net = LinearRegression(len(feature_names), out_size)

criterion = nn.MSELoss()
optimizer = optim.Adam(net.parameters(), lr=learning_rate, weight_decay=weight_decay)

if mode == 'autoregressive':
    predictor = AutoregressivePredictor(net, criterion, optimizer, device,
                                        n_ar=n_ar, predict_tendency=predict_tendency, k=k)
else:
    predictor = ParallelPredictor(net, criterion, optimizer, device,
                                  n_ar=n_ar, predict_tendency=predict_tendency)

## Train

In [ ]:
for epoch, train_losses, test_losses in predictor.fit(train_loader, test_loader, n_epochs):
    pass

In [ ]:
plt.plot(train_losses, label='train'); plt.plot(test_losses, '--', label='test')
plt.yscale('log'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend();

## Evaluate + save

In [ ]:
net.eval()
from model import _step, rollout
from experiment import integrate_tendency
with torch.no_grad():
    if mode == 'autoregressive':
        out_val = rollout(net, X_val.to(device), n_ar, predict_tendency).cpu()
    else:
        out_val = _step(net, X_val.to(device), n_ar, predict_tendency).cpu()

if target_tendency:
    # net predicts dM -> integrate from the true initial M, so drift accumulates
    dM = real_units(out_val, tmean, tstd).reshape(val_sel.n_cases, val_sel.n_time, -1)
    pred_val = integrate_tendency(dM, val_sel.y_state[:, :1, :])[..., 0]
    truth_val = val_sel.y_state[..., 0]
else:
    pred_val = invert_transform(real_units(out_val, tmean, tstd), target.transform)
    truth_val = invert_transform(val_sel.y, target.transform)

rmse = compute_rmse(pred_val.reshape(-1, 1), truth_val.reshape(-1, 1)).item()
print('val rmse:', rmse)

In [ ]:
config = PredictorConfig(
    run_id=run_id, mode=mode,
    dataset_path=f'{data_dir}/ePBL_paper_expanded_2283_corrected_transient.nc',
    val_dataset_path=f'{data_dir}/ePBL_paper_validation_2283_corrected_transient.nc',
    ar_features=[dict(name=s.name, lag=s.lag, transform=s.transform) for s in ar_features],
    forcings=[dict(name=s.name, lag=s.lag, transform=s.transform) for s in forcings],
    target=[dict(name=target.name, lag=target.lag, transform=target.transform)],
    feature_names=feature_names,
    network=network, n_neurons_list=n_neurons_list, activation=activation,
    n_ar=n_ar, predict_tendency=predict_tendency, target_tendency=target_tendency, k=k,
    learning_rate=learning_rate, weight_decay=weight_decay, loss=loss,
    n_epochs=epoch, random_state=random_state,
    train_batch_size=train_batch_size, test_batch_size=test_batch_size,
    feature_mean=fmean.tolist(), feature_std=fstd.tolist(),
    target_mean=tmean.tolist(), target_std=tstd.tolist(),
    train_loss_trajectory=train_losses, test_loss_trajectory=test_losses,
    n_time=train_sel.n_time, n_time_val=val_sel.n_time,
    n_cases=train_sel.n_cases, n_cases_val=val_sel.n_cases, rmse=rmse,
)
os.makedirs(run_dir, exist_ok=True)
torch.save(net.state_dict(), os.path.join(run_dir, 'weights.pt'))
config.save(run_dir)
print('saved', run_dir)